# Retrieval v5 — một luồng chung HS/Legal

Dev: 752 HS + 520 Legal. Test: 188 HS + 130 Legal. Chọn bằng Dev, khóa cấu hình rồi đánh giá Test qua Auto. Test đã được xem là hồi quy. Không dùng latency làm điều kiện chọn. Nhận diện ngữ nghĩa chỉ là metadata; không định tuyến hoặc lọc nguồn.


In [1]:
from pathlib import Path
import sys
import subprocess

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from rag.data import read, digest
from rag.db import connect

RUN = ROOT / "artifacts/experiments/run_unified_accuracy_v5"
RESUME_BENCHMARK = False
RUN_RUNTIME_DEMO = True


## 1. Thực nghiệm đã khóa

`RESUME_BENCHMARK=False` chỉ đọc kết quả đã có. Khi bật, runner resume protocol hiện có, sàng lọc M3/Qwen/Gemma, so sánh cascade trên Dev rồi kích hoạt policy và chấm Test. Không chạy đồng thời với runner đang hoạt động. Source/data đã đổi cần một protocol/lượt thực nghiệm mới.


In [2]:
if RESUME_BENCHMARK:
    subprocess.run([sys.executable, str(ROOT / "scripts/benchmark_unified_retrieval.py"), "run"], cwd=ROOT, check=True)

policy = read(RUN / "locked_policy.json")
assert policy == read(ROOT / "config/unified_retrieval.json"), "Runtime policy differs from Dev lock"
with connect() as conn:
    active = conn.execute("SELECT snapshot,profile FROM rag.active").fetchone()
assert active == {"snapshot": policy["snapshot"], "profile": policy["profile"]}
display(policy)


{'version': 'unified-source-v5',
 'candidate_pool': 160,
 'rrf_k': 60,
 'weights_name': 'balanced',
 'channel_weights': {'direct': 0.65,
  'context_lex': 0.125,
  'context_dense': 0.125,
  'neighbors': 0.1},
 'method': 'B1',
 'context_format': 'plain',
 'model': 'cascade',
 'rerank_pool': 100,
 'rank_weight': 0.85,
 'cascade_pool': 40,
 'cascade_weight': 0.5,
 'snapshot': 'a6717fc579f18c7a1dbac656',
 'profile': '3a2ce6e6d726a0f053748717',
 'intent_artifact': 'config/semantic_intent.json',
 'intent_hash': 'c895b8458aec664cea8f5fbe2d656c425378408516ea9a5513a60c850cf10e8a',
 'models': {'m3': {'model': 'BAAI/bge-reranker-v2-m3',
   'revision': '953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e',
   'backend': 'cross_encoder',
   'batch_size': 4},
  'qwen': {'model': 'Qwen/Qwen3-Reranker-0.6B',
   'revision': 'e61197ed45024b0ed8a2d74b80b4d909f1255473',
   'backend': 'qwen',
   'batch_size': 8},
  'gemma': {'model': 'BAAI/bge-reranker-v2-gemma',
   'revision': '1787044f8b6fb740a9de4557c3a12377f84d9e1

## 2. Kết quả Test thực tế

HS đo Top-10 HS8 chiếu từ cùng pool đã chấm; `mixed_*` đo vị trí trong danh sách nguồn hỗn hợp. Legal đòi mọi nhóm ID nguồn sơ cấp trong Top-10, gói nguồn bổ sung được báo riêng. Baseline v4 gọi đúng tác vụ đã biết; v5 dùng Auto chung. Không dùng Test để chọn lại cấu hình.


In [3]:
result = read(RUN / "test_metrics.json")
assert result["policy_hash"] == digest(policy)
display({k: v for k, v in result.items() if k != "rows"})
display(read(RUN / "test_comparison.json")["intervals"])
from IPython.display import Markdown
display(Markdown((RUN / "IMPLEMENTATION_REPORT_V5.md").read_text()))


{'objective': 0.8473800884709692,
 'HS': {'count': 188,
  'hit@1': 0.6808510638297872,
  'recall@5': 0.9042553191489362,
  'recall@10': 0.973404255319149,
  'mrr@10': 0.7832742316784869,
  'mixed_hit@1': 0.6276595744680851,
  'mixed_recall@10': 0.9361702127659575,
  'candidate_coverage': 0.9840425531914894,
  'rerank_coverage': 0.9787234042553191},
 'sections': {'XIV': {'count': 49,
   'hit@1': 0.5102040816326531,
   'recall@5': 0.9183673469387755,
   'recall@10': 0.9795918367346939,
   'mrr@10': 0.6937641723356008,
   'mixed_hit@1': 0.40816326530612246,
   'mixed_recall@10': 0.8571428571428571},
  'XVI': {'count': 85,
   'hit@1': 0.8117647058823529,
   'recall@5': 0.9411764705882353,
   'recall@10': 0.9882352941176471,
   'mrr@10': 0.8696078431372548,
   'mixed_hit@1': 0.7764705882352941,
   'mixed_recall@10': 0.9882352941176471},
  'XVII': {'count': 54,
   'hit@1': 0.6296296296296297,
   'recall@5': 0.8333333333333334,
   'recall@10': 0.9444444444444444,
   'mrr@10': 0.72860082304526

{'hs': {'hit@1': {'delta': 0.026595744680851064,
   'ci95': [-0.021742100261344738, 0.08064516129032258]},
  'recall@5': {'delta': 0.03723404255319149,
   'ci95': [-0.01098901098901099, 0.09138359171354088]},
  'recall@10': {'delta': 0.05851063829787234,
   'ci95': [0.01092896174863388, 0.12325098838494201]},
  'mrr@10': {'delta': 0.03954956095913543,
   'ci95': [0.005301376669171361, 0.0744106567747872]}},
 'qa': {'all_groups@10': {'delta': 0.023076923076923078,
   'ci95': [-0.007751937984496124, 0.061068702290076333]},
  'group_recall@10': {'delta': 0.023076923076923078,
   'ci95': [-0.007751937984496124, 0.061068702290076333]},
  'mrr@10': {'delta': 0.009325396825396825,
   'ci95': [-0.04192783859482333, 0.054831195398055824]}}}

# Báo cáo Retrieval chung HS/Legal — v5

## 1. Thay đổi luồng phục vụ

Auto dùng một thuật toán cho mọi câu hỏi: hybrid toàn nguồn → context BM25/dense → ứng viên từ quan hệ graph → pool chung theo ID → reranker/cascade chung → chứng cứ và shortlist HS8.
Nhận diện ngữ nghĩa là thông tin bổ trợ sau retrieval. `intent=product/qa` là hint hiển thị, không lọc hoặc đổi trọng số. Các API `adaptive_search`/`qa_search` và config v4 được giữ để tái lập đối chứng; Auto v5 không gọi router từ khóa và từ chối phục vụ khi chưa có policy chung đã kích hoạt. `legacy_routed_search` chỉ dùng để tái lập router v4.

Luồng v4: HS dùng B1/M3, pool 100/rerank 60, trọng số .35/.15/.35/.05/.10; QA dùng C10, pool 100/rerank 30, trọng số .25/.25/.10/.10/.30. Auto chọn nhánh bằng regex; chỉ đưa 48/130 câu Legal Test vào QA. Đây là tỷ lệ định tuyến, không phải Recall pháp lý.

V5 dùng pool nguồn chung 160 và cửa sổ rerank đã chọn bên dưới. So sánh v4/v5 là so sánh hai pipeline, thay cả tuyển ứng viên, biểu diễn context và xếp hạng; không thể quy toàn bộ mức thay đổi cho riêng model reranker hoặc việc bỏ router.

## 2. Cấu hình đã chọn chỉ bằng Dev

```json
{
  "version": "unified-source-v5",
  "candidate_pool": 160,
  "rrf_k": 60,
  "weights_name": "balanced",
  "channel_weights": {
    "direct": 0.65,
    "context_lex": 0.125,
    "context_dense": 0.125,
    "neighbors": 0.1
  },
  "method": "B1",
  "context_format": "plain",
  "model": "cascade",
  "rerank_pool": 100,
  "rank_weight": 0.85,
  "cascade_pool": 40,
  "cascade_weight": 0.5,
  "snapshot": "a6717fc579f18c7a1dbac656",
  "profile": "3a2ce6e6d726a0f053748717",
  "intent_artifact": "config/semantic_intent.json",
  "intent_hash": "c895b8458aec664cea8f5fbe2d656c425378408516ea9a5513a60c850cf10e8a",
  "models": {
    "m3": {
      "model": "BAAI/bge-reranker-v2-m3",
      "revision": "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e",
      "backend": "cross_encoder",
      "batch_size": 4
    },
    "qwen": {
      "model": "Qwen/Qwen3-Reranker-0.6B",
      "revision": "e61197ed45024b0ed8a2d74b80b4d909f1255473",
      "backend": "qwen",
      "batch_size": 8
    },
    "gemma": {
      "model": "BAAI/bge-reranker-v2-gemma",
      "revision": "1787044f8b6fb740a9de4557c3a12377f84d9e17",
      "backend": "flag_llm",
      "batch_size": 4,
      "last_token_only": true
    }
  },
  "instruction": "Retrieve cited source passages relevant to the query. For a product description, find matching tariff descriptions using their source hierarchy. For a legal question, find the source provisions that answer it. Do not assume unstated product attributes.",
  "max_length": 2048,
  "weights": {
    "dense": 0.35,
    "sparse": 0.15,
    "bm25": 0.35,
    "fts": 0.05,
    "exact": 0.1
  },
  "retrieval_config": {
    "model": "BAAI/bge-m3",
    "revision": "5617a9f61b028005a4858fdac845db406aefb181",
    "device": "cpu",
    "torch_num_threads": 4,
    "batch_size": 4,
    "max_length": 1024,
    "checkpoint_size": 64,
    "sparse": true,
    "candidate_pool": 50,
    "top_k": 10,
    "rrf_k": 60,
    "chunk_strategy": "contextual-v2",
    "evidence_limit": 30,
    "evidence_hops": 2,
    "seed": 20260926,
    "fusion_weights": {
      "dense": 1.0,
      "sparse": 1.0,
      "fts": 0.25
    },
    "reranker_model": "BAAI/bge-reranker-v2-m3",
    "reranker_revision": "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e",
    "rerank_pool": 30,
    "retrieval_version": "shared-source-graph-v2"
  }
}
```

Pool giữ cả HS4/HS6/HS8, Notes, GIR và điều khoản pháp lý. Không gộp các ID chứng cứ có cùng mã HS. Các nhánh và thông số áp dụng chung cho mọi Chương. B1 thêm phân cấp; B2 thêm Notes liên quan từ graph. Model và prompt được khóa; cache phân biệt revision, query, context, instruction và token budget.

Kênh graph lấy tổ tiên và Notes của các nguồn đã tìm được; chưa duyệt đầy đủ mọi quan hệ tham chiếu hoặc thực thi GIR. Các điều kiện trong code phục vụ schema, ngân sách token, backend model và kiểm tra lock; không có keyword → mã HS, phạt thuộc tính theo mã hoặc cấu hình theo Chương.

`matches` là Top-10 nguồn hỗn hợp. `hs_candidates` là Top-10 HS8 chiếu từ cùng pool đã chấm. `package` materialize các đoạn nguồn thực sự được trích trong context, có ngân sách lớn hơn Top-10. Điểm HS shortlist và điểm trên danh sách hỗn hợp được báo riêng. Điểm Legal chính đòi đúng mọi ID nguồn sơ cấp trong Top-10; không tính alias hay Notes đính kèm thay cho đoạn nguồn trực tiếp.

## 3. Thực nghiệm Dev

Dev: 752 HS + 520 QA = 1.272 câu. Screen cố định 62 câu, giữ toàn nhóm, cân bằng Phần HS và trải các loại QA. Sàng lọc theo chất lượng, không dùng latency. Hai model đơn tốt nhất được chấm toàn Dev; M3 và cascade M3 → Qwen cũng được chấm toàn Dev. Model bị loại ở screen chỉ có bằng chứng trên screen.

Tiêu chí chính: trung bình(HS Macro MRR@10, Legal strict all-groups@10). Tie-break: QA group recall, HS Macro Recall@5, ngân sách nhỏ hơn. Hai trọng số hybrid, B1/B2, pool 60/100, rank weight .85/1; cascade shortlist 20/40, weight .5/1. Phương án tốt nhất chỉ nằm trong không gian đã khảo sát, chưa là tối ưu toàn cục.

| Model | Phạm vi | Method / weights | Objective | HS Hit@1 | Legal đủ nhóm @10 |
|---|---|---|---:|---:|---:|
| m3 | Screen | B1 / balanced | 0.8045 | 62.5% | 86.7% |
| qwen | Screen | B1 / lexical | 0.8334 | 68.8% | 86.7% |
| gemma | Screen | B1 / lexical | 0.8437 | 62.5% | 93.3% |
| cascade | Toàn Dev | B1 / balanced | 0.8464 | 68.4% | 93.8% |
| gemma | Toàn Dev | B1 / lexical | 0.8325 | 66.5% | 92.7% |
| m3 | Toàn Dev | B1 / balanced | 0.8203 | 64.6% | 93.1% |
| qwen | Toàn Dev | B1 / lexical | 0.8450 | 67.7% | 93.1% |

Cấu hình thắng Dev: Macro MRR HS 0.7544, Legal đủ nhóm @10 93.8%.

## 4. Test qua Auto thực tế sau khi khóa

| Phạm vi HS | Số câu | Hit@1 v4 | Hit@1 v5 | Recall@5 v4 | Recall@5 v5 | MRR@10 v5 |
|---|---:|---:|---:|---:|---:|---:|
| XIV | 49 | 59.2% | 51.0% | 83.7% | 91.8% | 0.6938 |
| XVI | 85 | 78.8% | 81.2% | 92.9% | 94.1% | 0.8696 |
| XVII | 54 | 50.0% | 63.0% | 79.6% | 83.3% | 0.7286 |
| Toàn HS | 188 | 65.4% | 68.1% | 86.7% | 90.4% | 0.7833 |

HS hỗn hợp: Top-1 toàn danh sách nguồn đúng mã 62.8%; mã đúng trong Top-10 toàn nguồn 93.6%. Không được gọi các số shortlist ở bảng trên là Top-1 của toàn danh sách nguồn.

| Legal Test (130 câu) | v4 gọi QA trực tiếp | v5 Auto chung |
|---|---:|---:|
| Đủ mọi nhóm sơ cấp @10 | 90.8% | 93.1% |
| Group Recall@10 | 91.2% | 93.5% |
| MRR@10 | 0.7582 | 0.7675 |
| Gói nguồn bổ sung đủ nhóm | Khác quy trình mở rộng | 94.6% |

Coverage candidate pool v5: HS 98.4%, Legal group coverage 97.7%. Coverage cửa sổ rerank: HS 97.9%, Legal 96.5%.

## 5. Khoảng tin cậy thay đổi

Lấy mẫu ghép cặp nguyên cụm 2.000 lần; delta và CI dùng trung bình theo câu, không trộn với Macro. Baseline dùng tác vụ đúng đã biết; v5 dùng Auto cho cả hai.

| Tác vụ / chỉ số | Delta | 95% CI |
|---|---:|---|
| hs / hit@1 | +2.660 điểm % | [-2.174, +8.065] điểm % |
| hs / recall@5 | +3.723 điểm % | [-1.099, +9.138] điểm % |
| hs / recall@10 | +5.851 điểm % | [+1.093, +12.325] điểm % |
| hs / mrr@10 | +0.040 | [+0.005, +0.074] |
| qa / all_groups@10 | +2.308 điểm % | [-0.775, +6.107] điểm % |
| qa / group_recall@10 | +2.308 điểm % | [-0.775, +6.107] điểm % |
| qa / mrr@10 | +0.009 | [-0.042, +0.055] |

## 6. Nhận diện ngữ nghĩa và giới hạn

Logistic regression trên vector BGE-M3 của câu hỏi; C chọn bằng 5 Dev folds giữ nhóm, accuracy rồi log-loss. Model embedding/reranker không fine-tune. C=10. Test balanced accuracy=100.0%, confusion HS/QA=[[188, 0], [0, 130]]. Nhận diện không ảnh hưởng candidates hoặc ranking.

Các câu benchmark đã từng được xem. Test là hồi quy; kết quả chưa chứng minh chất lượng ở Chương mới hoặc câu thiếu thuộc tính. Thêm Chương cùng schema sẽ đi qua cùng thuật toán sau ingest/rebind snapshot. Tính đúng của áp dụng Notes/GIR và quyết định mã cuối vẫn cần kiểm chứng nguồn; retrieval không phải bộ suy luận pháp lý hoàn chỉnh.

Không dùng latency làm điều kiện chọn. Số latency trong test_metrics.json gồm query/model caches theo lịch sử chạy, chỉ mô tả lượt đo; không dùng làm SLA. Các artifact score/pool đầy đủ cho kiểm tra lại; config và nguồn không được chọn lại bằng Test.

Gemma dùng logits của token cuối, left padding và position_ids tường minh, tắt KV cache/hidden-state outputs không cần thiết. Cache layout này tách khỏi backend full logits cũ. Kiểm tra 4 cặp nguồn cho cùng thứ tự, chênh lệch logit tối đa 0.0078125 (gemma_equivalence.json); đây là kiểm tra backend nhỏ, không thay cho đánh giá chất lượng. Chỉ số screen/full/Test dùng layout mới; không suy ra tương đương số học tuyệt đối từ 4 cặp này.

## 7. Nghiên cứu và tái lập

[Sentence Transformers: Retrieve & Re-Rank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html) mô tả tuyển ứng viên trước khi rerank. [Qwen model card](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B) cung cấp prompt instruction và yes/no logits; v5 thay prompt chỉ dành cho sản phẩm bằng yêu cầu tìm chứng cứ chung. [Gemma model card](https://huggingface.co/BAAI/bge-reranker-v2-gemma) là căn cứ backend thử nghiệm. [Grouped CV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html) hỗ trợ giữ nhóm câu cùng họ trong đánh giá.

Runner duy nhất: `scripts/benchmark_unified_retrieval.py`. Protocol, source/profile binding, dataset SHA-256, pools, raw scores, screen selection, Dev selection, intent model, locked policy, Test runtime rows và paired intervals nằm trong thư mục này. Test runtime dùng `Retriever.routed_search`, không ép tác vụ từ gold.

## 8. Kiểm chứng runtime

6 câu Dev đối chiếu chính xác Top-10 nguồn và shortlist HS8 giữa runtime và replay; mỗi câu thử cả auto/product/qa. Mọi package ID đều là bản ghi nguồn thực; semantic intent không tham gia retrieval. Đây là kiểm tra tính nhất quán triển khai, không là tập đánh giá mới.

## 9. Kiểm thử triển khai

- pytest: PASS. 104 passed, 5 warnings in 52.27s
- ruff: PASS. 
- format: PASS. 
- mypy: PASS. 
- diff: PASS. 
- 01_build_knowledge_base.ipynb: PASS. 
- 02_test_retrieval.ipynb: PASS. 


## 3. Demo chung cho mọi câu hỏi

`matches`: Top-10 đoạn nguồn hỗn hợp. `hs_candidates`: shortlist HS8 từ cùng ranking. `package`: nguồn sơ cấp thực sự được trích trong context. `semantic_intent` không ảnh hưởng kết quả tìm kiếm. Không có kết luận HS tự động. Demo tắt mặc định vì cần nạp mô hình.


In [4]:
if RUN_RUNTIME_DEMO:
    from rag.embedding import Embedder
    from rag.retrieval import Retriever
    with connect() as conn:
        retriever = Retriever(conn, Embedder())
        for query in ["Ngọc trai nuôi cấy đã gia công", "Quy tắc phân loại hàng hóa chưa hoàn chỉnh được áp dụng như thế nào?"]:
            out = retriever.routed_search(query)
            display({"query":query, "routing":out["routing"], "semantic_intent":out["semantic_intent"], "sources":[{"id":r["id"],"code":r.get("code"),"locator":r.get("locator")} for r in out["matches"]], "hs_codes":[r["code"] for r in out["hs_candidates"]]})


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 22 files:   0%|                                | 0/22 [00:00<?, ?it/s]

Fetching 22 files: 100%|█████████████████████| 22/22 [00:00<00:00, 4256.21it/s]

Fetching 13 files:   0%|                                | 0/13 [00:00<?, ?it/s]

Fetching 13 files: 100%|█████████████████████| 13/13 [00:00<00:00, 3024.51it/s]

Fetching 14 files:   0%|                                | 0/14 [00:00<?, ?it/s]

Fetching 14 files: 100%|█████████████████████| 14/14 [00:00<00:00, 4047.44it/s]

{'query': 'Ngọc trai nuôi cấy đã gia công',
 'routing': {'status': 'UNIFIED_SOURCE_RETRIEVAL',
  'intent': 'knowledge',
  'hint': 'auto'},
 'semantic_intent': {'status': 'ADVISORY',
  'p_legal': 0.38802190792416963,
  'p_product': 0.6119780920758304,
  'intent': 'product',
  'used_for_retrieval': False,
  'model': 'bge-m3-query-embedding-logistic-regression'},
 'sources': [{'id': 'tt31:HS8:71012200',
   'code': '71012200',
   'locator': {'bbox': [102.0, 278.0, 361.0, 294.0],
    'audit': {'status': 'PASS',
     'lineage': {'graph_node_id': 'HS8:71012200',
      'hierarchy_node_id': 'HS8:71012200',
      'source_raw_record_id': 'XIV_SRC_P014',
      'structured_record_id': 'HS8:71012200'},
     'phase8_resolution': None,
     'resolved_issue_ids': []},
    'basis': 'PDF_CODE_ANCHOR',
    'sha256': 'b42bfc1cbd9ce4fe2f1ff2b8fb8300cd2e25e507d56a1e77472a24c8d9e6e2cd',
    'source': {'printed_page': 650,
     'row_bbox_pdf': {'x0': 20, 'x1': 825, 'y0': 278.5, 'y1': 295.0},
     'physical_pag

{'query': 'Quy tắc phân loại hàng hóa chưa hoàn chỉnh được áp dụng như thế nào?',
 'routing': {'status': 'UNIFIED_SOURCE_RETRIEVAL',
  'intent': 'knowledge',
  'hint': 'auto'},
 'semantic_intent': {'status': 'ADVISORY',
  'p_legal': 0.9936392077134685,
  'p_product': 0.006360792286531525,
  'intent': 'qa',
  'used_for_retrieval': False,
  'model': 'bge-m3-query-embedding-logistic-regression'},
 'sources': [{'id': 'gir:GIR_2_NOTE_009',
   'code': None,
   'locator': {'basis': 'PDF_TEXT_LAYER_COMPARISON',
    'pages': [8],
    'sha256': '9addd67d5feb3d05b4118ef3a9f9dec15854c9c53f41d070d8350c437d10d9d9',
    'document': '6_GIR.pdf',
    'context_pages': [8],
    'physical_pages': [6, 8, 10],
    'context_source_ids': ['GIR_2_NOTE_009'],
    'word_trigram_coverage': 1.0,
    'legacy_normalization_locator': {'pages': [8],
     'document': 'Phụ lục II - 6_GIR.pdf',
     'context_pages': [8],
     'context_source_ids': ['GIR_2_NOTE_009']}}},
  {'id': 'gir:GIR_2A',
   'code': None,
   'locator